# Notebook 04.5: Default LightGBM Baseline

This notebook is a supporting baseline-comparison notebook paired with Notebook 04. It mirrors the same overall preprocessing and holdout structure so that the model family, rather than the surrounding pipeline, remains the main point of comparison.

Its purpose is straightforward: provide a fair default LightGBM reference before the project commits to deeper XGBoost tuning.

## 1. Setup

This section imports the required libraries, defines the project paths, and fixes the random seed used throughout the notebook. Keeping the setup explicit helps maintain reproducibility and makes the comparison easier to interpret later.


In [ ]:
from pathlib import Path
import json
import sys
import warnings

import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
LOG_DIR = PROJECT_ROOT / 'logs'
MODEL_DIR = PROJECT_ROOT / 'models'
SUBMISSION_DIR = PROJECT_ROOT / 'submissions'

RANDOM_STATE = 42
TARGET = 'diagnosed_diabetes'
BASELINE_AUC = 0.6825
XGBOOST_DEFAULT_AUC = 0.7192

LOG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (12, 5)

print(f'Project root: {PROJECT_ROOT}')
print(f'Random seed: {RANDOM_STATE}')

## 2. Data Loading

We begin by loading the same training and test files used in the baseline XGBoost notebook. For consistency, `diagnosed_diabetes` is treated as the target variable and `id` is excluded from the predictive feature set. The purpose of this section is simply to confirm the dataset dimensions and establish the modeling target before any preprocessing is applied.


In [ ]:
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')

print(f'Train shape: {train.shape}')
print(f'Test shape: {test.shape}')
print(f'Target positive rate: {train[TARGET].mean():.4f}')

data_overview = pd.DataFrame(
    {
        'train_rows': [len(train)],
        'test_rows': [len(test)],
        'target': [TARGET],
        'id_used_as_feature': [False],
    }
)
data_overview

## 3. Preprocessing and Feature Representation

To preserve comparability with `04_xgboost_default`, the preprocessing pipeline is intentionally kept simple and aligned with the earlier notebook:

- drop `id` and the target from the feature matrix
- use the same categorical feature list
- apply `pd.get_dummies` for one-hot encoding
- avoid adding new feature engineering in this baseline notebook

This means any performance difference should mainly reflect the model choice rather than a change in data preparation.


In [ ]:
categorical_cols = [
    'gender',
    'ethnicity',
    'education_level',
    'income_level',
    'smoking_status',
    'employment_status',
]

X = train.drop(['id', TARGET], axis=1)
y = train[TARGET]
X_encoded = pd.get_dummies(X, columns=categorical_cols)

print(f'Encoded feature count: {X_encoded.shape[1]}')
X_encoded.head()

## 4. Train / Validation Split

The notebook uses the same holdout strategy as `04_xgboost_default`:

- `test_size=0.2`
- `random_state=42`
- `stratify=y`

This alignment is important because it keeps the validation environment fixed. As a result, the observed difference in AUC can be interpreted primarily as a model comparison under the same split, rather than an artifact of a different evaluation design.


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f'Train set: {X_train.shape}, Validation set: {X_val.shape}')

## 5. Train Default LightGBM

We fit `LGBMClassifier` with default or near-default settings, adding only a few minimal compatibility arguments for binary classification and AUC evaluation. The intention is to keep this notebook clearly in the baseline category, so the result can be compared directly with the default XGBoost baseline rather than with a tuned LightGBM variant.


In [ ]:
lgbm_default = lgb.LGBMClassifier(
    objective='binary',
    metric='auc',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
)

lgbm_default.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
)

y_pred_proba = lgbm_default.predict_proba(X_val)[:, 1]
y_pred = lgbm_default.predict(X_val)

current_auc = roc_auc_score(y_val, y_pred_proba)

print(f'Baseline Decision Tree AUC: {BASELINE_AUC:.4f}')
print(f'Default XGBoost AUC (from Experiment 1 summary): {XGBOOST_DEFAULT_AUC:.4f}')
print(f'Default LightGBM AUC: {current_auc:.4f}')
print(f'Improvement over baseline: {current_auc - BASELINE_AUC:+.4f}')
print(f'Difference vs default XGBoost: {current_auc - XGBOOST_DEFAULT_AUC:+.4f}')

print('\nClassification report:')
print(classification_report(y_val, y_pred))

## 6. Results

This section reports the validation AUC and then examines two standard diagnostic views:

- the validation learning curve, which shows how the evaluation metric changes over boosting iterations
- LightGBM feature importance, which gives a compact view of which predictors the model relies on most strongly

A practical reading of these outputs is to look for both overall performance and model behavior: the AUC indicates validation quality, while the plots provide context for how that result is achieved.


In [ ]:
results = lgbm_default.evals_result_
metric_name = list(results['valid_0'].keys())[0]
epochs = len(results['valid_0'][metric_name])
x_axis = range(0, epochs)

importance = pd.DataFrame(
    {
        'feature': X_train.columns,
        'importance': lgbm_default.feature_importances_,
    }
).sort_values('importance', ascending=False).head(15)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(x_axis, results['valid_0'][metric_name], 'b-', label='Validation AUC')
axes[0].axhline(y=BASELINE_AUC, color='r', linestyle='--', label=f'Baseline ({BASELINE_AUC:.4f})')
axes[0].axhline(y=XGBOOST_DEFAULT_AUC, color='g', linestyle='--', label=f'XGBoost default ({XGBOOST_DEFAULT_AUC:.4f})')
axes[0].set_xlabel('Iterations')
axes[0].set_ylabel('AUC')
axes[0].set_title('LightGBM Learning Curve')
axes[0].legend()
axes[0].grid(True)

axes[1].barh(importance['feature'], importance['importance'])
axes[1].set_xlabel('Importance')
axes[1].set_title('Top 15 LightGBM Feature Importance')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print('Top 10 most important features:')
print(importance.head(10).to_string(index=False))

## 7. Comparison with `04_xgboost_default`

The comparison table places the default LightGBM result beside the recorded default XGBoost result from `logs/exp1_summary.txt`. Because both notebooks use the same 80/20 stratified holdout logic and a closely aligned preprocessing path, this table serves as a straightforward apples-to-apples baseline comparison.

The main quantity to inspect is the validation AUC difference. A small gap suggests broadly comparable baseline behavior, while a larger gap suggests that one model family is the stronger untuned starting point under this specific setup.

In [ ]:
comparison_df = pd.DataFrame(
    [
        {
            'model': 'XGBoost default',
            'split_method': '80/20 stratified holdout',
            'validation_auc': XGBOOST_DEFAULT_AUC,
            'key_note': 'Recorded from logs/exp1_summary.txt',
        },
        {
            'model': 'LightGBM default',
            'split_method': '80/20 stratified holdout',
            'validation_auc': current_auc,
            'key_note': 'Same setup as 03_xgboost_default, different model only',
        },
    ]
)
comparison_df

## 8. Save Artifacts

All exported files use dedicated `lightgbm_default_*` names so they remain separate from the rest of the notebook series. This keeps the notebook safe to run without overwriting later workflow artifacts or other comparison outputs.

In [ ]:
summary_json_path = LOG_DIR / 'lightgbm_default_summary.json'
summary_md_path = LOG_DIR / 'lightgbm_default_summary.md'
feature_importance_path = LOG_DIR / 'lightgbm_default_feature_importance.csv'
val_predictions_path = LOG_DIR / 'lightgbm_default_val_predictions.csv'
plot_path = LOG_DIR / 'lightgbm_default_plots.png'
model_path = MODEL_DIR / 'lightgbm_default.pkl'
submission_path = SUBMISSION_DIR / 'lightgbm_default.csv'

feature_importance_full = pd.DataFrame(
    {
        'feature': X_train.columns,
        'importance': lgbm_default.feature_importances_,
    }
).sort_values('importance', ascending=False)
feature_importance_full.to_csv(feature_importance_path, index=False)

val_predictions = pd.DataFrame(
    {
        'y_true': y_val.to_numpy(),
        'y_pred_proba': y_pred_proba,
        'y_pred_label': y_pred,
    }
)
val_predictions.to_csv(val_predictions_path, index=False)

summary = {
    'experiment_name': 'Default LightGBM baseline',
    'model': 'LGBMClassifier',
    'split_method': '80/20 stratified holdout',
    'random_state': RANDOM_STATE,
    'baseline_auc': BASELINE_AUC,
    'xgboost_default_auc': XGBOOST_DEFAULT_AUC,
    'lightgbm_default_auc': float(current_auc),
    'auc_vs_baseline': float(current_auc - BASELINE_AUC),
    'auc_vs_xgboost_default': float(current_auc - XGBOOST_DEFAULT_AUC),
    'train_shape': list(X_train.shape),
    'validation_shape': list(X_val.shape),
    'top_features': feature_importance_full.head(10).to_dict(orient='records'),
}

with open(summary_json_path, 'w', encoding='utf-8') as handle:
    json.dump(summary, handle, indent=2)

summary_md = '\n'.join([
    '# Default LightGBM Summary',
    '',
    '## Metrics',
    f'- Baseline Decision Tree AUC: {BASELINE_AUC:.4f}',
    f'- Default XGBoost AUC: {XGBOOST_DEFAULT_AUC:.4f}',
    f'- Default LightGBM AUC: {current_auc:.4f}',
    f'- Difference vs default XGBoost: {current_auc - XGBOOST_DEFAULT_AUC:+.4f}',
    '',
    '## Split',
    f'- Train shape: {X_train.shape}',
    f'- Validation shape: {X_val.shape}',
    '',
    '## Notes',
    '- This notebook mirrors 03_xgboost_default as closely as possible.',
    '- The only intentional model change is replacing XGBoost with default LightGBM.',
]) + '\n'
summary_md_path.write_text(summary_md, encoding='utf-8')

fig.savefig(plot_path, dpi=180, bbox_inches='tight')
joblib.dump(lgbm_default, model_path)

X_test = test.drop(['id'], axis=1)
X_test_encoded = pd.get_dummies(X_test, columns=categorical_cols)
missing_cols = set(X_train.columns) - set(X_test_encoded.columns)
for col in missing_cols:
    X_test_encoded[col] = 0
X_test_encoded = X_test_encoded[X_train.columns]

test_pred = lgbm_default.predict_proba(X_test_encoded)[:, 1]
submission = pd.DataFrame({'id': test['id'], 'diagnosed_diabetes': test_pred})
submission.to_csv(submission_path, index=False)

print(f'Saved summary JSON: {summary_json_path}')
print(f'Saved summary Markdown: {summary_md_path}')
print(f'Saved feature importance CSV: {feature_importance_path}')
print(f'Saved validation predictions CSV: {val_predictions_path}')
print(f'Saved plot image: {plot_path}')
print(f'Saved model: {model_path}')
print(f'Saved submission: {submission_path}')

## 9. Conclusion

This notebook provides a clean default LightGBM baseline under the same holdout setup as `04_xgboost_default`.

- If default LightGBM performs below default XGBoost here, that suggests XGBoost is the stronger untuned baseline under this specific split and preprocessing setup.
- If default LightGBM performs similarly or better, that suggests LightGBM is also a competitive starting point for future work.
- In either case, this notebook should be read as a supporting baseline comparison rather than as the project's final modeling direction.

A practical interpretation is that this notebook helps position LightGBM relative to the existing XGBoost baseline while keeping the broader experiment story focused on comparable evidence rather than isolated model claims.